# 最小圆问题

**类别：** 选址

来源： [https://www.hexaly.com/templates/smallest-circle-problem](https://www.hexaly.com/templates/smallest-circle-problem)


## 问题

**Smallest Circle Problem**（也称为 Minimum Covering Circle Problem 或 Smallest Enclosing Circle Problem）是一个计算几何问题，其目的是在欧几里得平面上计算包含给定点集的最小圆。

该问题是一个 [Facility Location Problem](https://www.hexaly.com/example/facility-location-problem-flp)（1-Center Problem）的实例，其中需要为新设施选择一个位置，以便为多个客户提供服务，并最小化任何客户到达该新设施所需的最远距离。

	

### 学到的建模原则

- 添加 [float decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#floating-point-decisions) 来建模圆心的坐标
- 使用 [非线性算子 ‘sqrt’ and ‘pow’](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算圆的半径
- 了解 Hexaly Optimizer 的建模风格：[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)


## 数据

数据文件的格式如下：

- 第一行：点的数量
- 接下来几行：对于每个点，其 x 和 y 坐标


## 模型

Smallest Circle Problem 的 Hexaly 模型使用两个 float 决策变量，分别表示圆心的横坐标和纵坐标。使用非线性算子 **sqrt**、**min** 和 **pow**，我们可以将圆的半径计算为其圆心到各点的所有距离的最小值。事实上，半径不需要是一个决策变量。由于其值可以从其他决策变量中计算得到，它只是一个中间表达式。

目标函数是最小化圆的半径。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python smallest_circle.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)


def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read instance data
    #
    file_it = iter(read_integers(sys.argv[1]))
    # Number of points
    nb_points = next(file_it)

    # Point coordinates
    coord_x = [None] * nb_points
    coord_y = [None] * nb_points

    coord_x[0] = next(file_it)
    coord_y[0] = next(file_it)

    # Minimum and maximum value of the coordinates of the points
    min_x = coord_x[0]
    max_x = coord_x[0]
    min_y = coord_y[0]
    max_y = coord_y[0]

    for i in range(1, nb_points):
        coord_x[i] = next(file_it)
        coord_y[i] = next(file_it)
        if coord_x[i] < min_x:
            min_x = coord_x[i]
        else:
            if coord_x[i] > max_x:
                max_x = coord_x[i]
        if coord_y[i] < min_y:
            min_y = coord_y[i]
        else:
            if coord_y[i] > max_y:
                max_y = coord_y[i]

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # x, y are respectively the abscissa and the ordinate of the origin of the circle
    x = model.float(min_x, max_x)
    y = model.float(min_y, max_y)

    # Distance between the origin and the point i
    radius = [(x - coord_x[i]) ** 2 + (y - coord_y[i]) ** 2 for i in range(nb_points)]

    # Minimize the radius r
    r = model.sqrt(model.max(radius))
    model.minimize(r)

    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 6

    optimizer.solve()

    #
    # Write the solution in a file
    #
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as f:
            f.write("x=%f\n" % x.value)
            f.write("y=%f\n" % y.value)
            f.write("r=%f\n" % r.value)
